In [2]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path
from torch.utils.data import DataLoader
from spectre.datasets import RFUAVDataset
import torch
from torch import nn, optim
from torchvision.models import resnet18

transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(),
    ToTensorV2(),
])

ds = RFUAVDataset(root=Path("/datasets"), transform=transform)
loader = DataLoader(ds, batch_size=4, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = resnet18(weights="DEFAULT")
model.fc = nn.Linear(model.fc.in_features, len(ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

Device: cuda
Epoch 1/10 - Loss: 1.5637 - Acc: 0.4000
Epoch 2/10 - Loss: 0.6786 - Acc: 0.9000
Epoch 3/10 - Loss: 0.5754 - Acc: 0.9000
Epoch 4/10 - Loss: 0.1426 - Acc: 1.0000
Epoch 5/10 - Loss: 0.1683 - Acc: 1.0000
Epoch 6/10 - Loss: 0.1094 - Acc: 1.0000
Epoch 7/10 - Loss: 0.1302 - Acc: 1.0000
Epoch 8/10 - Loss: 0.0272 - Acc: 1.0000
Epoch 9/10 - Loss: 0.1204 - Acc: 1.0000
Epoch 10/10 - Loss: 0.0502 - Acc: 1.0000
